In [3]:

"""
Oil and Vinegar (OV) / Unbalanced Oil and Vinegar (UOV) Dijital İmzalama Şeması
================================================================================

Bu modül, çok değişkenli polinom tabanlı (ÇDPT)
kriptografinin BigField ailesinden farklı olarak, doğrudan temel sonlu
cisim F_q üzerinde çalışan Oil and Vinegar (OV) ailesine ait UOV (Unbalanced
Oil and Vinegar) imza şemasının öğretici bir SageMath simülasyonunu
içermektedir.

Teorik Arka Plan
-----------------
Patarin, 1997 yılında Matsumoto-Imai (MI) şemasına yönelttiği doğrusallaştırma
saldırısından esinlenerek "The oil and vinegar signature scheme" başlıklı
çalışmasında imzalama amaçlı yeni bir çok değişkenli polinom tabanlı
kriptosistem önermiştir. HFE ve MI gibi önceki BigField şemalarının aksine
OV şeması, cisim genişlemesi üzerinde tanımlanan dönüşümlere değil, doğrudan
temel sonlu cisim F_q üzerindeki işlemlere dayanır. Bu sebeple OV şemaları,
BigField ailesine kıyasla hem hesaplamaları daha küçük cisimlerde
yapabilme kolaylığı hem de çok daha hızlı çalışabilme avantajı sunar.

OV şemasının temel fikri, n = v + o adet değişkeni iki ayrık gruba
ayırmaktır:

    - Vinegar değişkenleri (v adet): x_0, ..., x_{v-1}
    - Oil değişkenleri (o adet):     x_v, ..., x_{n-1}

Gizli merkez dönüşüm F, bu iki değişken grubunu kullanarak o adet homojen
kuadratik polinomdan oluşur; ancak bu polinomların YAPISI özenle
kısıtlanmıştır: hiçbir polinomda iki Oil değişkeninin çarpımı (Oil x Oil)
YER ALMAZ. Yani merkez dönüşüm polinomları yalnızca

    Vinegar x Vinegar,  Vinegar x Oil

biçiminde kuadratik terimler içerebilir; Oil x Oil terimleri kesinlikle
sıfır katsayılıdır. Bu yapısal kısıtlama, merkez dönüşümün matris
temsilinde n x n boyutundaki matrisin sağ alt o x o bloğunun SIFIR matrisi
olması anlamına gelir:

        [ V x V bloğu   |  V x O bloğu ]
    M = [---------------+--------------]
        [ (simetriği)   |   0 (o x o)  ]

Bu kısıtlamanın kriptografik gerekçesi, imzalama sürecinde ortaya çıkar:
Vinegar değişkenlerine RASTGELE sayısal değerler atandığında, merkez
dönüşümün polinomları Oil değişkenlerine göre DOĞRUSAL hale gelir (çünkü
geriye yalnızca "sabit x Oil" biçimindeki terimler kalır, Oil x Oil terimi
zaten yoktur). Böylece n bilinmeyenli, o denklemli doğrusal olmayan bir
sistemi çözmek yerine, yalnızca o bilinmeyenli (Oil değişkenleri) DOĞRUSAL
bir denklem sistemi çözülerek bir ön-görüntü (imza) bulunabilir. Bu, OV
ailesinin "gizli tuzak kapısı" (trapdoor) mekanizmasının özüdür.

Açık anahtar, gizli merkez dönüşüm F ile gizli tersinir afin dönüşüm T'nin
bileşkesi olarak P = F o T biçiminde elde edilir. T dönüşümü, merkez
dönüşüm özel Vinegar/Oil blok yapısını saldırgandan gizleyerek açık
anahtarı, görünürde rastgele bir çok değişkenli kuadratik polinom sistemine
dönüştürür.

Dengeli (v = o) ve Dengesiz (Unbalanced, v > o) Varyantlar
------------------------------------------------------------
Patarin'in orijinal 1997 önerisinde v = o (dengeli OV) seçilmiştir; ancak
1999 yılında Kipnis, Patarin ve Goubin tarafından yayımlanan bir çalışmada,
Kipnis-Shamir'in 1998'de dengeli OV şemasına yönelik geliştirdiği polinom
zamanlı gizli anahtar kurtarma saldırısının, v > o (Unbalanced/Dengesiz
OV, kısaca UOV) seçilerek etkisiz hale getirilebileceği gösterilmiştir. Bu
nedenle günümüzde OV ailesinin pratikte kullanılan tüm varyantları
(Rainbow, MAYO, QR-UOV, SNOVA gibi Modern UOV yapıları dahil) v > o
koşuluna dayanır.

Algoritmanın Genel Akışı
-------------------------
1. Anahtar Üretimi:
   - Gizli tersinir afin dönüşüm T (n x n matris + öteleme vektörü) rastgele
     seçilir.
   - Her biri sağ alt o x o bloğu sıfır olan o adet simetrik/üst-üçgen
     matris (Central_Map_Matrices) rastgele üretilir; bu matrisler gizli
     merkez dönüşüm F'nin kuadratik kısmını temsil eder. Doğrusal ve sabit
     terimler de rastgele eklenir.
   - Açık anahtar, P = F o T bileşkesinin hem POLİNOM düzeyinde (sembolik
     yerine koyma ile) hem de kuadratik form MATRİSİ düzeyinde
     (M_pub = T^T * M_central * T dönüşümü ile) hesaplanmasıyla elde edilir.

2. İmzalama:
   - Vinegar değişkenlerine rastgele değerler atanır.
   - Bu değerler merkez dönüşüm polinomlarında yerine konularak, Oil
     değişkenlerine göre DOĞRUSAL bir denklem sistemi (A * o_vec = b)
     kurulur.
   - Bu doğrusal sistem çözülerek Oil değişkenlerinin değerleri bulunur
     (sistem çözümsüzse farklı Vinegar değerleriyle tekrar denenir).
   - Vinegar ve Oil değerleri birleştirilerek elde edilen z vektörüne
     T dönüşümünün TERSİ uygulanarak nihai imza elde edilir.

3. Doğrulama:
   - İmza, açık anahtar polinomlarında doğrudan yerine konularak P(imza)
     hesaplanır ve mesaj vektörü ile karşılaştırılır.

Bu implementasyon; yüksek lisans tezi kapsamında OV/UOV şemasının kavramsal
ve matematiksel yapısını somutlaştırmak amacıyla eğitim/araştırma amaçlı
geliştirilmiştir. Prodüksiyon ortamlarında kullanılmak üzere tasarlanmamış
olup sabit zamanlı (constant-time) işlemler veya yan kanal saldırılarına
karşı önlemler içermemektedir.

-------------
Bu kod SageMath ortamında çalıştırılmak üzere tasarlanmıştır ve sonlu
cisimler (GF), çok değişkenli polinom halkaları (PolynomialRing), matris/
vektör cebiri gibi SageMath'in yerleşik simgesel/sayısal cebir araçlarına
dayanır.

Referans: Tez, Bölüm 6

"""

from sage.all import *


class UOVSignatureScheme:
    """
    Oil and Vinegar (OV) / Unbalanced Oil and Vinegar (UOV) dijital
    imzalama şemasının uçtan uca, öğretici bir referans implementasyonu.

    Bu sınıf; gizli/açık anahtar üretimi, imzalama ve doğrulama
    süreçlerini kapsayan tam bir OV/UOV yaşam döngüsünü modelleyen
    durumsal (stateful) bir nesne sunar. Sınıfın her bir örneği; sabit
    (q, v, o) parametre setine bağlı olarak kendi sonlu cismini, çok
    değişkenli polinom halkasını ve (üretildiğinde) gizli/açık anahtar
    bileşenlerini üzerinde taşır.

    Matematiksel Kurulum
    ---------------------
    - F_q  : Temel sonlu cisim (karakteristik/asal güç q). BigField
      ailesindeki HFE tipi şemaların aksine, OV/UOV şemasının TÜM
      işlemleri doğrudan bu cisim üzerinde yürütülür; ayrı bir cisim
      genişlemesine ihtiyaç duyulmaz.
    - R = F_q[x_0, ..., x_{n-1}] : n = v + o değişkenli, F_q katsayılı
      çok değişkenli polinom halkası; hem gizli merkez dönüşüm hem de
      açık anahtar polinomlarının yaşadığı ortak uzay.
    - Değişken bölünmesi: x_0, ..., x_{v-1} Vinegar değişkenleri;
      x_v, ..., x_{n-1} Oil değişkenleridir.

    Parametreler
    ------------
    q : int
        Temel sonlu cismin eleman sayısı (F_q). Genellikle küçük bir
        asal sayı veya asal kuvveti olarak seçilir.
    v : int
        Vinegar değişkenlerinin sayısı. Kipnis-Shamir saldırısına karşı
        dayanıklılık için v > o (Unbalanced/Dengesiz yapı) seçilmesi
        önerilir; v = o seçilirse Patarin'in orijinal (kırılmış) dengeli
        OV şemasına dönülür.
    o : int
        Oil değişkenlerinin sayısı; aynı zamanda açık anahtardaki
        (ve merkez dönüşümdeki) denklem/polinom sayısına eşittir.

    Öznitelikler (Attributes)
    --------------------------
    n : int
        Toplam değişken sayısı (n = v + o); açık anahtarın girdi
        uzayının boyutu ve gizli T dönüşümünün tanım kümesinin boyutu.
    F : FiniteField
        Temel sonlu cisim F_q.
    R : MPolynomialRing
        n değişkenli, F_q katsayılı çok değişkenli polinom halkası.
    vars : vector
        R halkasının üreteçlerinden (x_0, ..., x_{n-1}) oluşan vektör;
        polinom işlemlerinde (kuadratik form hesaplarında) doğrudan
        matris çarpımına sokulabilmesi için vektör biçiminde tutulur.
    Central_Map_Matrices : list of Matrix
        Gizli merkez dönüşüm F'nin her bir F_k bileşenine ait kuadratik
        form matrisi (sağ alt o x o bloğu sıfır olan n x n matrisler).
    Central_Map_Polys : list of MPolynomial
        Gizli merkez dönüşüm F'nin o adet polinom bileşeni.
    Public_Key_Quadratic_Matrices : list of Matrix
        Açık anahtarın her bir P_k bileşenine ait kuadratik form matrisi;
        M_pub = T_mat^T * M_central * T_mat dönüşümüyle elde edilir.
    Public_Key_Polys : list of MPolynomial
        Açık anahtarı oluşturan o adet çok değişkenli kuadratik polinom.
    T_mat, T_vec : Matrix, vector
        Gizli afin dönüşüm T(x) = T_mat * x + T_vec (n x n, tersinir).

    Notlar
    ------
    Kurucu metot (__init__), temel cebirsel altyapıyı (sonlu cisim, çok
    değişkenli polinom halkası) kurar; anahtar bileşenleri (T_mat,
    Central_Map_Matrices, Public_Key_Polys vb.) başlangıçta boş/None
    olarak ayarlanır ve generate_keys() metodu çağrılana kadar
    doldurulmaz. Bu ayrım, imzalama/doğrulama metotlarının anahtarların
    üretilip üretilmediğini _check_keys_generated() ile denetleyebilmesine
    imkân tanır.
    """

    def __init__(self, q, v, o):
        """
        Oil and Vinegar / Unbalanced Oil and Vinegar imza şeması için
        öğretici SageMath simülasyonu.

        q: Sonlu cismin eleman sayısı
        v: Vinegar değişkenlerinin sayısı
        o: Oil değişkenlerinin sayısı
        n = v + o: Toplam değişken sayısı
        """
        self.q = q
        self.v = v
        self.o = o
        self.n = v + o

        # F_q temel sonlu cismi: OV/UOV şemasının BigField (HFE, MI)
        # ailesinden en temel farkı, herhangi bir cisim genişlemesine
        # ihtiyaç duymadan doğrudan bu cisim üzerinde çalışmasıdır.
        self.F = GF(q, 'a')
        # n = v + o değişkenli, F_q katsayılı çok değişkenli polinom
        # halkası; hem gizli merkez dönüşüm hem de açık anahtar
        # polinomlarının ortak tanım kümesidir.
        self.R = PolynomialRing(self.F, self.n, 'x')
        self.vars = vector(self.R, self.R.gens()) # x vektörü (x0..xn-1)

        print("\n" + "="*70)
        print(f"UOV SİSTEMİ: TAM DETAYLI POLİNOM VE MATRİS ANALİZİ")
        print(f"Parametreler: GF({q}), v={v}, o={o} (n={self.n})")
        print(f"Değişkenler: Vinegar=[x0..x{v-1}], Oil=[x{v}..x{self.n-1}]")
        print("="*70)

        # Gizli ve açık anahtar bileşenleri, generate_keys() çağrılana
        # kadar boş/None olarak tutulur. Bu, sınıfın anahtar üretiminden
        # önceki durumunu açıkça ayırt eder ve _check_keys_generated()
        # tarafından denetlenir.
        self.Central_Map_Matrices = []
        self.Central_Map_Polys = []
        self.Public_Key_Quadratic_Matrices = []
        self.Public_Key_Polys = []
        self.T_mat = None; self.T_vec = None


    def _check_keys_generated(self):
        """
        Açık ve gizli anahtarların üretilip üretilmediğini kontrol eder.

        Bu iç kontrol metodu, sign() ve verify() gibi anahtarlara bağımlı
        işlemlerin, generate_keys() çağrılmadan önce yanlışlıkla
        çalıştırılmasını (ve tanımsız/None nesneler üzerinde hataya yol
        açmasını) önlemek amacıyla kullanılır.

        İstisnalar (Raises)
        --------------------
        RuntimeError
            T_mat henüz None ise veya gizli/açık anahtar polinom
            listeleri (Central_Map_Polys, Public_Key_Polys) boşsa
            fırlatılır.
        """
        if self.T_mat is None or len(self.Central_Map_Polys) == 0 or len(self.Public_Key_Polys) == 0:
            raise RuntimeError("Önce generate_keys() çağrılmalıdır.")


    def generate_keys(self):
        """
        UOV şemasının gizli anahtarını (T dönüşümü ve merkez dönüşüm F)
        rastgele üretir ve bunlardan açık anahtarı (Public_Key_Polys) hem
        polinom hem de matris düzeyinde türetir.

        Algoritmanın Adımları
        -----------------------
        1. Afin Dönüşüm T:
           n x n boyutunda TERSİNİR rastgele bir matris (T_mat) ve rastgele
           bir öteleme vektörü (T_vec) seçilir. Tersinirlik koşulu, T'nin
           bir bijeksiyon olmasını (dolayısıyla imzalama sırasında T^{-1}'in
           var olmasını) garanti eder. T dönüşümü, merkez dönüşümün özel
           Vinegar/Oil blok yapısını saldırgandan gizleyen "maskeleme"
           katmanıdır.

        2. Merkez Dönüşüm F (Gizli):
           o adet kuadratik form matrisi M_k (k = 0, ..., o-1) üretilir.
           Her M_k matrisinin (i, j) girdisi, YALNIZCA i veya j indislerinden
           en az biri bir Vinegar değişkenine (i < v veya j < v) karşılık
           geliyorsa rastgele bir değer alır; hem i hem de j bir Oil
           değişkenine karşılık geliyorsa (i >= v ve j >= v) bu girdi
           SIFIR bırakılır. Bu, UOV şemasının tanımlayıcı kısıtlamasıdır:
           merkez dönüşümde Oil x Oil kuadratik terimi bulunmaz. Her M_k
           matrisinden, x^T * M_k * x kuadratik formu ile rastgele
           doğrusal ve sabit terimlerin toplanmasıyla F_k polinomu elde
           edilir.

        3. Açık Anahtar P (P = F o T):
           - Önce x değişkenleri T dönüşümü ile y = T_mat * x + T_vec
             ifadesine taşınır.
           - Her bir merkez dönüşüm polinomu F_k, bu y ifadeleri sembolik
             olarak yerine konularak P_k = F_k(T(x)) açık anahtar
             polinomuna dönüştürülür (POLİNOM BİLEŞKESİ).
           - Aynı dönüşüm, MATRİS düzeyinde de gerçekleştirilir: P_k'nın
             kuadratik formunu temsil eden matris,
                 M_pub_k = T_mat^T * M_k * T_mat
             eşitliği ile hesaplanır. Bu eşitlik, x^T * M_k * x kuadratik
             formunda x yerine T_mat * x + T_vec konulduğunda, ikinci
             dereceden (kuadratik) kısmın yalnızca T_mat'a bağlı olarak
             bu bilineer dönüşümle taşındığı gerçeğine dayanır; afin
             öteleme vektörü T_vec ise yalnızca doğrusal ve sabit
             terimleri etkiler.

        Bu iki paralel hesaplama (polinom bileşkesi ve matris dönüşümü),
        aynı açık anahtarın iki farklı temsilini sunarak kuadratik form
        matrislerinin rolünün somut biçimde gözlemlenmesini sağlar.

        Yan Etkiler
        -----------
        self.T_mat, self.T_vec, self.Central_Map_Matrices,
        self.Central_Map_Polys, self.Public_Key_Quadratic_Matrices ve
        self.Public_Key_Polys öznitelikleri bu metot tarafından
        (yeniden) doldurulur. Metot birden fazla kez çağrılırsa, önceki
        anahtar bileşenleri temizlenip yeniden üretilir. Ayrıca süreç
        boyunca ayrıntılı ara çıktılar ekrana yazdırılır.

        Dönüş Değeri
        ------------
        None
            Üretilen anahtar bileşenleri doğrudan nesne öznitelikleri
            olarak saklanır; metot bir değer döndürmez.
        """
        # Eğer generate_keys() birden fazla kez çağrılırsa,
        # eski anahtar bileşenleri temizlenir.
        self.Central_Map_Matrices = []
        self.Central_Map_Polys = []
        self.Public_Key_Quadratic_Matrices = []
        self.Public_Key_Polys = []

        print("\n" + "*"*30 + " ADIM 1: ANAHTAR ÜRETİMİ " + "*"*30)

        # --- 1. AFİN DÖNÜŞÜM T ---
        # T, n x n boyutunda TERSİNİR rastgele bir doğrusal dönüşüm ve
        # rastgele bir öteleme vektöründen oluşur. Tersinirlik, imzalama
        # sürecinde T^{-1}'in hesaplanabilir olmasını garanti eder; bu
        # dönüşüm gizli anahtarın "maskeleme" bileşenidir ve merkez
        # dönüşümün Vinegar/Oil blok yapısını saldırgandan saklar.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible(): break
        self.T_vec = random_vector(self.F, self.n)

        print(f"\n[1.1] Gizli Afin Dönüşüm T (x -> x'):")
        print(f"   Matris M_T:\n{self.T_mat}")
        print(f"   Vektör c_T: {self.T_vec}")

        # --- 2. MERKEZ DÖNÜŞÜM (F) ---
        print(f"\n[1.2] Gizli Merkez Dönüşüm F(x) (Matris ve Polinom Hali)")
        print(f"      KURAL: Matrislerin sağ alt {self.o}x{self.o} bloğu SIFIR olmalı!")

        for k in range(self.o):
            # A) Matris Oluşturma (Üst Üçgen)
            # M_k, kuadratik formu x^T * M_k * x temsil eden n x n boyutlu
            # bir matristir; yalnızca üst üçgen (i <= j) girdiler
            # doldurulur, böylece her kuadratik terim x_i*x_j yalnızca
            # bir kez temsil edilir.
            M_k = matrix(self.F, self.n, self.n)

            # UOV kuralı:
            # Oil-oil kuadratik terimleri bulunmaz.
            # Yani i ve j aynı anda oil değişkeni ise M_k[i,j] = 0 kalır.
            # Bu kısıtlama, merkez dönüşümün "tuzak kapısı" özelliğinin
            # temelidir: Vinegar değişkenleri sabitlendiğinde geriye
            # yalnızca Oil değişkenlerine göre DOĞRUSAL terimler kalmasını
            # sağlar.
            for i in range(self.n):
                for j in range(i, self.n):
                    if not (i >= self.v and j >= self.v):
                        M_k[i, j] = self.F.random_element()

            self.Central_Map_Matrices.append(M_k)

            # B) Polinom Türetme: x^T M x + L(x) + c
            # Kuadratik kısım, x vektörünün M_k matrisi ile bilineer
            # formu (x^T * M_k * x) olarak hesaplanır; buna rastgele
            # katsayılı bir doğrusal kısım (L(x)) ve rastgele bir sabit
            # terim (c) eklenerek merkez dönüşümün k. bileşen polinomu
            # tamamlanır.
            quad_part = self.vars * M_k * self.vars

            lin_part = self.R(0)
            for i in range(self.n):
                lin_part += self.F.random_element() * self.vars[i]
            const_part = self.F.random_element()

            poly = quad_part + lin_part + const_part
            self.Central_Map_Polys.append(poly)

            print(f"\n   --- F_{k} (Gizli) ---")
            print(f"   Polinom: {poly}")
            print(f"   Matris M_{k} (Blok Yapısına Dikkat):")
            print(M_k)

        # --- 3. AÇIK ANAHTAR (P) ---
        print(f"\n[1.3] Açık Anahtar P(x) = F(T(x))")
        print(f"      Kuadratik kısım için matris dönüşümü: M_pub = T^T * M_central * T")

        # T(x)
        # x değişkenleri, gizli afin dönüşüm T aracılığıyla
        # y = T_mat * x + T_vec biçiminde "karıştırılır". Bu adım,
        # saldırganın açık anahtardan doğrudan orijinal x değişkenlerine
        # (ve dolayısıyla merkez dönüşümün özel blok yapısına)
        # erişimini engelleyen temel gizleme mekanizmasıdır.
        y_vec = self.T_mat * self.vars + self.T_vec
        sub_dict = {self.vars[i]: y_vec[i] for i in range(self.n)}

        for k in range(self.o):
            # A) Polinom Bileşkesi
            # Merkez dönüşüm polinomu F_k, y = T(x) ifadeleri sembolik
            # olarak yerine konularak açık anahtar polinomuna
            # dönüştürülür: P_k(x) = F_k(T(x)).
            p_poly = self.Central_Map_Polys[k].subs(sub_dict)
            self.Public_Key_Polys.append(p_poly)

            # B) Matris Bileşkesi (Kuadratik Form Transferi)
            # P(x) = F(T(x)) olduğundan kuadratik kısım M_T^T * Q * M_T biçiminde dönüşür.
            # Afin öteleme c_T ise lineer ve sabit terimleri etkiler.
            M_pub_quad = self.T_mat.transpose() * self.Central_Map_Matrices[k] * self.T_mat
            self.Public_Key_Quadratic_Matrices.append(M_pub_quad)

            print(f"\n   --- P_{k} (Açık) ---")
            print(f"   Polinom: {p_poly}")
            print(f"   Matris M_pub_{k} Açık Anahtarın Kuadratik Kısmının Bir Temsilidir:")
            print(f"   Not: Afin dönüşümden dolayı açık anahtar polinomunda lineer ve sabit terimler de değişir.")
            print(M_pub_quad)

    def sign(self, message_vec):
        """
        Verilen mesaj (hash) vektörü için UOV gizli anahtarını kullanarak
        bir dijital imza üretir.

        Algoritmanın Adımları
        -----------------------
        UOV şemasının imzalama sürecindeki temel fikir, merkez dönüşümün
        Vinegar değişkenlerine rastgele sayısal değerler atandığında Oil
        değişkenlerine göre DOĞRUSAL hale gelmesidir (çünkü merkez
        dönüşümde Oil x Oil terimi hiç bulunmaz). Bu sayede, doğrusal
        OLMAYAN bir denklem sistemi çözmek yerine, çok daha kolay çözülebilen
        doğrusal bir denklem sistemi çözülerek bir ön-görüntü bulunur.

        1. Vinegar Seçimi:
           F_q üzerinde rastgele v adet Vinegar değeri (v_vals) seçilir.

        2. Doğrusal Sistem Kurulumu:
           Seçilen Vinegar değerleri, her bir merkez dönüşüm polinomu F_k
           içinde YERİNE KONUR (f_partial = F_k(v_vals sabit, Oil)).
           Vinegar x Vinegar terimleri sabit bir sayıya, Vinegar x Oil
           terimleri ise Oil değişkenlerine göre doğrusal terimlere
           dönüştüğünden (ve Oil x Oil terimi zaten yoktur), f_partial
           artık yalnızca Oil değişkenlerine göre DOĞRUSAL bir ifadedir.
           Bu doğrusal ifadenin her bir Oil değişkenine ait katsayısı
           (monomial_coefficient ile) ve sabit terimi çıkarılarak
                 A * oil_vals = b
           biçiminde bir doğrusal denklem sistemi (A: katsayılar matrisi,
           b: hedef vektör) oluşturulur; burada b_k = message_vec[k] -
           f_partial'ın sabit terimi olarak hesaplanır.

        3. Doğrusal Sistemin Çözümü:
           A * oil_vals = b sistemi Sage'in solve_right metoduyla
           çözülmeye çalışılır. Sistem çözümsüzse (A tekil/rank
           yetersizse), farklı bir rastgele Vinegar seçimiyle işlem
           tekrarlanır (retry mekanizması; en fazla 100 deneme).

        4. Birleştirme ve Ters Dönüşüm:
           Bulunan Oil değerleri, seçilen Vinegar değerleriyle
           birleştirilerek z = (Vinegar || Oil) vektörü elde edilir. z,
           aslında T(x) = z denklemindeki z'ye karşılık geldiğinden,
           orijinal imza x = T^{-1} * (z - T_vec) hesaplanarak elde edilir.

        Parametreler
        ------------
        message_vec : vector
            Uzunluğu o olan, F_q üzerinde tanımlı mesaj (hash) vektörü.

        Dönüş Değeri
        ------------
        vector veya None
            Başarılı olursa, uzunluğu n olan imza vektörü (F_q üzerinde);
            100 deneme içinde doğrusal sistem çözülemezse None döner.

        İstisnalar (Raises)
        --------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse (_check_keys_generated
            aracılığıyla) fırlatılır.
        ValueError
            message_vec'in uzunluğu o değilse fırlatılır.
        """
        self._check_keys_generated()

        if len(message_vec) != self.o:
            raise ValueError(f"Mesaj vektörünün uzunluğu o={self.o} olmalıdır.")

        print("\n" + "*"*30 + " ADIM 2: İMZALAMA SÜRECİ " + "*"*30)
        print(f"   İmzalanacak Mesaj: {message_vec}")
        attempts = 0
        while attempts < 100:
            attempts += 1
            print(f"\n   --- Deneme {attempts} ---")

            # A) Vinegar Seçimi
            # F_q üzerinde rastgele seçilen bu değerler, merkez dönüşümü
            # Oil değişkenlerine göre doğrusal bir sisteme indirgemek için
            # "dondurulacak" (sabitlenecek) değişkenlerdir.
            v_vals = random_vector(self.F, self.v)
            print(f"   1. Rastgele Vinegar (v): {v_vals}")

            # B) Lineer Sistem Kurulumu
            # F_k(v_sabit, o) = msg_k
            # Vinegar değerleri sabitlendiğinde, merkez dönüşüm
            # polinomlarının Oil x Oil terimi içermemesi sayesinde geriye
            # yalnızca Oil değişkenlerine göre DOĞRUSAL bir ifade kalır.

            coeffs_matrix = []
            consts_vector = []
            oil_vars = self.vars[self.v:]

            v_dict = {self.vars[i]: v_vals[i] for i in range(self.v)}

            print(f"   2. Oil Değişkenleri İçin Denklem Sistemi:")

            for k in range(self.o):
                # Polinomda vinegar'ları yerine koy
                f_partial = self.Central_Map_Polys[k].subs(v_dict)

                # Lineer form: Sum(a_i * o_i) + C = msg_k
                # Sum(a_i * o_i) = msg_k - C
                # Her bir Oil değişkeninin doğrusal katsayısı çekilerek
                # katsayılar matrisinin k. satırı oluşturulur; sabit terim
                # ise hedef vektörün k. bileşenini belirlemek için mesaj
                # değerinden çıkarılır.
                eq_coeffs = []
                for o_var in oil_vars:
                    eq_coeffs.append(f_partial.monomial_coefficient(o_var))
                coeffs_matrix.append(eq_coeffs)

                constant_term = f_partial.constant_coefficient()
                target = message_vec[k] - constant_term
                consts_vector.append(target)

                # Denklemi yazdır
                eq_str = " + ".join([f"{c}*{v}" for c, v in zip(eq_coeffs, oil_vars)])
                print(f"      F_{k} => {eq_str} = {target}")

            # Matris Çözümü
            # A * oil_vals = b biçimindeki doğrusal sistem, oluşturulan
            # katsayılar matrisi (A) ve hedef vektör (b) kullanılarak
            # çözülür. Bu adım, UOV imzalama sürecinin hesaplama
            # açısından en kritik ve aynı zamanda EN HIZLI adımıdır;
            # HFE tipi şemalardaki kök bulma işleminin aksine burada
            # yalnızca doğrusal cebir gerekmektedir.
            A = matrix(self.F, coeffs_matrix)
            b = vector(self.F, consts_vector)

            print(f"      Matris A:\n{A}")
            print(f"      Vektör b: {b}")

            try:
                oil_vals = A.solve_right(b)
                print(f"   3. [BAŞARILI] Oil Çözümü: {oil_vals}")

                # C) Birleştirme
                # Rastgele seçilen Vinegar değerleri ile bulunan Oil
                # değerleri birleştirilerek z = T(x) denklemindeki z
                # vektörü elde edilir.
                z_vec = vector(self.F, list(v_vals) + list(oil_vals))
                print(f"   4. Birleştirilmiş Vektör z: {z_vec}")

                # D) T^-1
                # z = T_mat * x + T_vec olduğundan, x = T_mat^{-1} *
                # (z - T_vec) ile orijinal imza (gizli anahtarla üretilmiş
                # ön-görüntü) hesaplanır.
                signature = self.T_mat.inverse() * (z_vec - self.T_vec)
                print(f"   5. T^-1 Uygulandı. İMZA (x): {signature}")

                return signature

            except ValueError:
                print("   3. [BAŞARISIZ] Lineer sistem çözülemedi. Yeni vinegar seçiliyor...")

        return None

    def verify(self, message, signature):
        """
        Verilen (mesaj, imza) ikilisinin, UOV açık anahtarına göre geçerli
        olup olmadığını denetler.

        Teorik Gerekçe
        ---------------
        Doğrulama süreci, imzalamanın aksine GİZLİ anahtara (T dönüşümü ve
        merkez dönüşüm F) ihtiyaç duymaz; yalnızca açık anahtar
        polinomlarının (self.Public_Key_Polys) doğrudan değerlendirilmesine
        dayanır. Bu asimetri (imzalamanın doğrusal sistem çözümü ve
        Vinegar rastgeleliğine bağlı, doğrulamanın ise doğrudan polinom
        değerlendirmesi olması), çok değişkenli polinom tabanlı imza
        şemalarının ortak karakteristik özelliğidir ve doğrulamanın
        pratikte çok hızlı gerçekleştirilmesini sağlar.

        Algoritmanın Adımları
        -----------------------
        1. Anahtarların üretilip üretilmediği ve girdi vektörlerinin
           (message, signature) uzunluklarının sırasıyla o ve n olduğu
           denetlenir.
        2. İmza vektörünün her bileşeni, ilgili değişkenlere
           (x_0, ..., x_{n-1}) atanarak açık anahtarın her bir polinomu
           P_k için P_k(signature) değeri hesaplanır.
        3. Elde edilen değerler vektörü check_vec = P(imza), orijinal
           mesaj vektörü (message) ile karşılaştırılır.

        Parametreler
        ------------
        message : vector
            Uzunluğu o olan, doğrulanacak orijinal mesaj (hash) vektörü.
        signature : vector
            sign() metodundan elde edilen, uzunluğu n olan imza vektörü.

        Dönüş Değeri
        ------------
        bool
            check_vec == message ise True (imza geçerli), aksi halde
            False (imza geçersiz).

        İstisnalar (Raises)
        --------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse (_check_keys_generated
            aracılığıyla) fırlatılır.
        ValueError
            message'ın uzunluğu o değilse veya signature'ın uzunluğu n
            değilse fırlatılır.
        """
        self._check_keys_generated()

        if len(message) != self.o:
            raise ValueError(f"Mesaj vektörünün uzunluğu o={self.o} olmalıdır.")

        if len(signature) != self.n:
            raise ValueError(f"İmza vektörünün uzunluğu n={self.n} olmalıdır.")

        print("\n" + "*"*30 + " ADIM 3: DOĞRULAMA " + "*"*30)
        # İmza vektörünün her bileşeni, açık anahtar polinomlarındaki
        # ilgili x_i değişkenine atanır ve her polinom bu değerler
        # altında değerlendirilerek P(imza) vektörü elde edilir.
        val_dict = {self.vars[i]: signature[i] for i in range(self.n)}
        check = [p.subs(val_dict) for p in self.Public_Key_Polys]
        check_vec = vector(self.F, check)

        print(f"   P(İmza) = {check_vec}")
        print(f"   Mesaj   = {message}")
        # Doğrulama koşulu: hesaplanan P(imza) değeri, orijinal mesaj
        # vektörü ile birebir eşleşiyorsa imza kriptografik olarak
        # geçerlidir.
        return check_vec == message



In [4]:
# ============================================================================
# SENARYO
# ============================================================================
# Bu blok, yukarıda tanımlanan UOVSignatureScheme sınıfının tam bir yaşam
# döngüsünü (anahtar üretimi -> imzalama -> doğrulama) örneklendirmek
# amacıyla çalıştırılan bir DEMO/TEST senaryosudur.
print("\n\n" + "*"*60)
print(">>> UOV DETAYLI SİMÜLASYON <<<")
print("*"*60)

# v: vinegar değişkenlerinin sayısı
# o: oil değişkenlerinin sayısı
# n = v + o
# v = o durumunda dengeli Oil and Vinegar yapısı,
# v > o durumunda ise Unbalanced Oil and Vinegar (UOV) yapısı elde edilir.
my_q = 5
my_v = 6
my_o = 3

uov = UOVSignatureScheme(q=my_q, v=my_v, o=my_o)
uov.generate_keys()

msg = random_vector(uov.F, my_o)
sig = uov.sign(msg)

if sig is not None:
    if uov.verify(msg, sig):
        print("\nSONUÇ: GEÇERLİ.")
    else:
        print("\nSONUÇ: GEÇERSİZ.")
else:
    print("\nSONUÇ: İmza üretilemedi.")




************************************************************
>>> UOV DETAYLI SİMÜLASYON <<<
************************************************************

UOV SİSTEMİ: TAM DETAYLI POLİNOM VE MATRİS ANALİZİ
Parametreler: GF(5), v=6, o=3 (n=9)
Değişkenler: Vinegar=[x0..x5], Oil=[x6..x8]

****************************** ADIM 1: ANAHTAR ÜRETİMİ ******************************

[1.1] Gizli Afin Dönüşüm T (x -> x'):
   Matris M_T:
[1 2 0 3 3 3 2 2 1]
[2 0 3 0 4 3 0 2 2]
[0 3 4 4 4 1 0 1 2]
[4 2 2 1 2 1 0 1 1]
[3 0 3 2 0 0 1 0 3]
[0 4 2 4 2 0 4 4 0]
[0 4 4 4 3 3 0 3 0]
[4 3 3 4 1 0 1 0 2]
[4 0 0 1 1 1 1 1 4]
   Vektör c_T: (3, 0, 3, 1, 3, 3, 4, 2, 4)

[1.2] Gizli Merkez Dönüşüm F(x) (Matris ve Polinom Hali)
      KURAL: Matrislerin sağ alt 3x3 bloğu SIFIR olmalı!

   --- F_0 (Gizli) ---
   Polinom: -2*x0*x1 - x0*x2 - 2*x1*x2 - 2*x2^2 + x0*x3 + 2*x1*x3 + x2*x3 + 2*x3^2 - 2*x0*x4 + 2*x1*x4 + 2*x2*x4 + x4^2 - 2*x0*x5 - 2*x1*x5 - x2*x5 + x3*x5 - 2*x4*x5 + 2*x5^2 + x0*x6 + x1*x6 + x2*x6 + 2*x4*x6 -